# Week 4, day 2 (afternoon) — dbt on Snowflake: from raw to data marts

The WeCloudData **Create a dbt Project** lab and the **dbt Fundamentals**
lecture, as one notebook that runs **entirely inside Snowflake**.

You build a complete, tested dbt project — sources, staging, snapshots, a
**Type 6** slowly-changing dimension, a **star schema** on surrogate keys,
business-driven **seeds**, Great-Expectations-style quality checks, **unit
tests**, a **MetricFlow semantic layer**, and **lineage** across every layer —
then hand it to Snowflake to execute as a native `DBT PROJECT` object.

## How it works

| Cell type | Does |
|---|---|
| **SQL** | the warehouse work: schemas, raw tables, loading, `EXECUTE DBT PROJECT`, inspecting results, scheduling |
| **Python** | writes the dbt project files, and uploads them to a stage |

**There are no credentials in this notebook.** `get_active_session()` gives you
the session you are already authenticated in, and dbt runs as the executing role.
Nothing to configure, nothing to leak.

## Before you start

- A role that can create a database, schemas, stages, tasks and a `DBT PROJECT`,
  and a warehouse attached to this notebook.
- The two CSVs to hand: `products.csv` (1,214 rows) and `sales.csv` (100,000
  rows), from the lab's `create_dbt_project_datasets.zip`. Question 5 uploads
  them through the Snowsight stage UI.

## The layers you are building

```
RAW  ──▶  STG  ──▶  EDW  ──▶  MARTS
 │         │         │          │
 │         │         │          ├─ rpt_*   hand-written SQL models
 │         │         │          └─ mart_*  exported from MetricFlow saved queries
 │         │         │
 │         │         └─ dim_product_t6 (Type 6) / dim_store / dim_date / fct_sales
 │         └─ stg_product_incr / stg_sales (views) + the snapshots
 └─ product / sales  (the two CSVs)
                    + seeds: store_master, category_targets
```

New to dbt? The day's README has a **concepts primer** — model, source, ref,
seed, snapshot, materialization, macro, test — read that first.

Work the questions in order: each builds on the objects the earlier ones made.


---

## Instructor notes — running this in class

**Everything below is the worked solution.** Cells are in dependency order: run
them top to bottom and each one has what it needs.

### Before the session

1. **A role and warehouse.** You need to create a database, schemas, stages,
   tasks and a `DBT PROJECT`. Attach a warehouse to this notebook, or the setup
   cell reports `CURRENT_WAREHOUSE()` as null.
2. **Have the two CSVs to hand** — `products.csv` (1,214 rows) and `sales.csv`
   (100,000 rows). Question 7 uploads them by hand; that is the one step no cell
   can do for you.
3. **Decide the packages route now, not mid-demo.** `dbt deps` (Question 44)
   needs outbound internet, which Snowflake blocks by default. Either have an
   ACCOUNTADMIN set up an external access integration for `hub.getdbt.com`
   beforehand, or vendor `dbt_packages/` into the upload. If neither is ready,
   Question 44 will fail in front of the class.
4. **Do a full dry run.** The first `build` takes a few minutes on 100k rows.

### The three manual steps

Everything else is a cell you run. These are not:

| Where | What you do |
|---|---|
| Question 7 | upload the two CSVs to `RAW.LOAD_STAGE` via **Data → Databases → DEMO_DB → RAW → Stages → LOAD_STAGE → + Files** |
| Question 42 | if vendoring packages, put `dbt_packages/` in the upload |
| Question 58 | **tear it down** — suspends the task and drops the lab objects, so nothing keeps billing |

### The beats worth slowing down on

- **Question 25** — the Type 6 dimension. Three CTEs, one per view. This is the
  conceptual centre of the day.
- **Question 28** — the fact table: grain stated explicitly, only keys and
  measures, delete+insert rather than merge.
- **Question 53** — change a product, re-snapshot, rebuild, and watch the Type 6
  columns diverge. This is the payoff; do not skip it for time.
- **Question 56** — `GET_DDL` on what dbt built. You never wrote `CREATE TABLE`;
  here is the DDL anyway. A good closing beat.
- **Questions 47–48** — lineage in both directions. Ask the room "what breaks if
  I change `stg_sales`?" before running it.

### Timing

Parts A–B (~25 min) are account setup and writing the project; move briskly.
Part C (seeds, snapshots, Type 6, star schema) is the substance — budget half
the session. Parts D–I are ~10 minutes each. Leave five minutes for Question 58,
the teardown.

### About the output

This notebook ships **without stored output**: it was written against no
Snowflake account, so nothing here is a recorded run. Each answer states what it
*should* return. Execute it once yourself before class so the outputs are real
and yours.


## A dbt primer — read this first

Everything below assumes this vocabulary. If you already know dbt, skip to Part A.

### What dbt is

Modern pipelines are **ELT**: data is *loaded* into the warehouse first, then
*transformed* inside it. dbt owns that **T**. It moves no data and has no runtime
of its own — it compiles your SQL and asks Snowflake to run it.

You write `SELECT` statements. dbt works out the order, wraps each in the right
`CREATE TABLE` / `CREATE VIEW` / `MERGE`, runs your tests, and builds the
documentation and lineage graph.

What it replaces: a folder of numbered `.sql` scripts where logic is
copy-pasted, dependencies live in someone's head, and "is this number right?" has
no answer.

### The objects

| Concept | What it is | Here |
|---|---|---|
| **Model** | a `.sql` file holding one `SELECT`; dbt materializes it | `stg_sales`, `dim_product_t6`, `fct_sales` |
| **Source** | a raw table dbt did *not* build, declared so it can be referenced and tested | `raw.product`, `raw.sales` |
| **`ref()`** | how one model reads another — this is what builds the DAG | everywhere |
| **`source()`** | how a model reads a raw table | staging models |
| **Seed** | a CSV in the project, loaded by `dbt seed` | `store_master`, `category_targets` |
| **Snapshot** | records how a *mutable* source changes over time | `product_snapshot` |
| **Test** | an assertion about the data; failures stop the build | `unique`, `not_null`, `relationships` |
| **Unit test** | an assertion about a model's *logic*, against mock rows | the Type 6 scenario |
| **Macro** | a Jinja function returning SQL | `to_active_flag` |
| **Package** | an installable library of macros and tests | `dbt_utils`, `dbt_expectations` |
| **Metric** | a business definition written once, in the semantic layer | `margin_rate` |

**Never hardcode a table name — always `ref()` or `source()`.** That one habit is
what gives dbt the dependency graph; build order, lineage and impact analysis all
follow from it.

### Materializations

| Materialization | dbt does | Use when |
|---|---|---|
| **view** (default) | `CREATE VIEW` each run | light transformation, no storage, always fresh |
| **table** | `CREATE TABLE` each run | queried often, rebuild is affordable |
| **incremental** | insert/update only new rows | full rebuild too slow |
| **ephemeral** | nothing — inlined as a CTE | a small step one or two models need |

**Rule of thumb:** start with a view; when it is slow to query make it a table;
when the table is slow to build make it incremental.

### Incremental models

Three parts, always: a `config` marking it incremental, an `is_incremental()`
block holding a **cutoff filter**, and the filter itself (usually against
`this`, the model's existing table). First run builds everything; later runs
process only rows past the watermark.

Two strategies appear here:

- **`merge`** — update matching rows, insert the rest. For keyed staging rows
  (`stg_product_incr`).
- **`delete+insert`** — delete the matching keys, then insert. For recomputed
  **aggregates**, which must be *replaced* rather than merged (`fct_sales`).

### Slowly-changing dimensions, and why Type 6

A source row changes and the old value is lost — unless you capture it.

| Type | Behaviour |
|---|---|
| **Type 1** | overwrite; only "now" exists |
| **Type 2** | a new row per version, with validity dates |
| **Type 3** | keep a `previous_*` column |
| **Type 6** | **1 + 2 + 3 together** (1 × 2 × 3 = 6) |

`dim_product_t6` carries all three on every row:

| View | Column | Answers |
|---|---|---|
| Type 2 | `category_name` | what was it **at the time**? |
| Type 1 | `current_category_name` | re-state history under **today's** value |
| Type 3 | `previous_category_name` | what did it **change from**? |

So the same fact row can be reported "as was" or "as is" without touching the
fact table. The **snapshot** supplies the Type 2 raw material
(`dbt_valid_from` / `dbt_valid_to`); the model derives the other two.

### Star schema and surrogate keys

A **fact** surrounded by **dimensions**, joined on **surrogate keys**.

```
                  dim_date
                      │
   dim_product_t6 — fct_sales — dim_store
```

The fact holds only keys and **measures**, at a stated **grain** (one row per
date / product version / store) — every measure must be additive at that grain.
Descriptions live in the dimensions, so a category rename never touches the fact.

A **surrogate key** is a warehouse-generated id with no business meaning
(`dbt_utils.generate_surrogate_key` hashes `prod_key` + `valid_from`). Why not
just `prod_key`? In an SCD it is **not unique** — a product has one row per
version. The surrogate key is unique per *version*, which is what lets the fact
point at exactly the right one.

### The layers

| Layer | Schema | Contains | Materialized |
|---|---|---|---|
| **raw** | `RAW` | the two loaded CSVs; dbt only reads these | — |
| **staging** | `STG` | 1:1 with sources: rename, cast, select. No joins or aggregation | views |
| **edw** | `EDW` | conformed dimensions and facts — the star schema | tables |
| **marts** | `MARTS` | business-facing reports | tables |

Each layer is a **folder** *and* a **schema** — which is what makes the lineage
readable and lets you build one layer at a time.

### Tests

- A **data test** runs against the warehouse; it compiles to a query returning
  offending rows, so zero rows = pass. The four generic ones are `unique`,
  `not_null`, `accepted_values`, `relationships`.
- **dbt-expectations** (the dbt port of Great Expectations) adds range,
  distribution, shape and freshness checks — the defects that pass every
  column-level test and still make a report wrong.
- A **unit test** feeds a model mock rows and asserts exact output. It checks
  *logic*, with no warehouse data — ideal for SCD rules.

### The commands

Inside Snowflake every one of these runs as
`EXECUTE DBT PROJECT <name> ARGS = '<command>'`:

```
deps            install packages
seed            load the CSV seeds
run             build models only
snapshot        build snapshots only (stateful — advances history)
test            run tests only
build           seeds + snapshots + models + tests, in DAG order   <- schedule this
ls --select +x  what feeds x?      ls --select x+   what breaks if x changes?
build --full-refresh               rebuild incrementals from scratch
```

`build` stops a downstream model when an upstream test fails, so bad data does
not propagate. That is why it is the command you schedule.


Run this cell first, every session. It picks up the session you are already
authenticated in — no account, user or password anywhere — and defines the names
every later cell uses. If `CURRENT_WAREHOUSE()` comes back null, attach a
warehouse to the notebook before going on.

In [ ]:
from snowflake.snowpark.context import get_active_session
from pathlib import Path

session = get_active_session()

# Everything the project needs, in one place. Change these if your account uses
# different names; every cell below reads them.
DB    = "DEMO_DB"
WH    = session.get_current_warehouse().strip('"')
ROLE  = session.get_current_role().strip('"')
PROJ  = "/tmp/demo"          # where the dbt project is assembled in this notebook
STAGE = f"{DB}.RAW.DBT_PROJECT_STAGE"

session.sql("SELECT CURRENT_ROLE(), CURRENT_WAREHOUSE(), CURRENT_VERSION()").show()
print("database :", DB)
print("warehouse:", WH)
print("role     :", ROLE)

## PART A — the environment

A Snowflake notebook already *is* connected: `get_active_session()` hands you the
session for the role, warehouse and database you are running as. There is no
`profiles.yml`, no account locator, no password — anywhere in this notebook.

That is the single biggest difference from running dbt on a laptop, and it is why
this whole lab fits in one notebook.

### Question 1

Create the **warehouse** this lab runs on, and check what privileges your
role has. Skip the `CREATE WAREHOUSE` if you already have one attached.

In [ ]:
-- The compute. XSMALL is plenty for 100k rows, and AUTO_SUSPEND keeps the
-- credit burn near zero between cells.
CREATE WAREHOUSE IF NOT EXISTS COMPUTE_WH
    WAREHOUSE_SIZE     = 'XSMALL'
    AUTO_SUSPEND       = 60      -- seconds idle before it parks itself
    AUTO_RESUME        = TRUE
    INITIALLY_SUSPENDED = TRUE;

USE WAREHOUSE COMPUTE_WH;

-- Who am I, and what can I do? Everything below needs CREATE DATABASE.
SELECT CURRENT_ROLE()      AS my_role,
       CURRENT_WAREHOUSE() AS my_warehouse,
       CURRENT_ACCOUNT()   AS my_account;

SHOW GRANTS TO ROLE IDENTIFIER(CURRENT_ROLE());

`AUTO_SUSPEND = 60` matters in a classroom: a warehouse left running bills by the second. `INITIALLY_SUSPENDED` means creating it costs nothing until the first query. If `SHOW GRANTS` does not include `CREATE DATABASE`, switch to a role that has it before going further.

### Question 2

**Optional, ACCOUNTADMIN only.** Create a dedicated role for the lab and grant
it what it needs. Skip this if you are already running as `SYSADMIN` or
similar — the rest of the notebook does not depend on it.

In [ ]:
-- A least-privilege role, rather than doing coursework as ACCOUNTADMIN.
USE ROLE ACCOUNTADMIN;

CREATE ROLE IF NOT EXISTS DBT_LAB_ROLE;

-- Compute, and the ability to make the objects this lab creates.
GRANT USAGE, OPERATE ON WAREHOUSE COMPUTE_WH TO ROLE DBT_LAB_ROLE;
GRANT CREATE DATABASE ON ACCOUNT             TO ROLE DBT_LAB_ROLE;

-- Give it to yourself, and to SYSADMIN so the hierarchy stays sane.
GRANT ROLE DBT_LAB_ROLE TO USER IDENTIFIER(CURRENT_USER());
GRANT ROLE DBT_LAB_ROLE TO ROLE SYSADMIN;

-- USE ROLE DBT_LAB_ROLE;   -- uncomment to switch, then re-run the cell above
SHOW GRANTS TO ROLE DBT_LAB_ROLE;

Doing coursework as ACCOUNTADMIN is the habit worth breaking early. Note dbt will run as whatever role executes the project, so whichever role you settle on must own — or be granted on — every schema in the next question.

### Question 3

Create the database and the five schemas, one per layer.

In [ ]:
-- RAW    the landing zone: the two CSVs, untouched
-- STG    staging (1:1 with raw) and the snapshots
-- EDW    the star schema: conformed dimensions and facts
-- MARTS  business-facing reports
-- SEED   dbt seeds (version-controlled reference data)
CREATE DATABASE IF NOT EXISTS DEMO_DB;

USE DATABASE DEMO_DB;

CREATE SCHEMA IF NOT EXISTS RAW;
CREATE SCHEMA IF NOT EXISTS STG;
CREATE SCHEMA IF NOT EXISTS EDW;
CREATE SCHEMA IF NOT EXISTS MARTS;
CREATE SCHEMA IF NOT EXISTS SEED;

SHOW SCHEMAS IN DATABASE DEMO_DB;

One schema per layer. dbt creates the objects *inside* them, but not the database itself — that is always yours to make.

### Question 4

Create the two raw tables, matching the CSV headers exactly.

In [ ]:
USE DATABASE DEMO_DB;
USE SCHEMA RAW;

CREATE OR REPLACE TABLE RAW.PRODUCT (
    PROD_KEY          NUMBER,
    PROD_NAME         VARCHAR,
    VOL               FLOAT,
    WGT               FLOAT,
    BRAND_NAME        VARCHAR,
    STATUS_CODE       NUMBER,
    STATUS_CODE_NAME  VARCHAR,
    CATEGORY_KEY      NUMBER,
    CATEGORY_NAME     VARCHAR,
    SUBCATEGORY_KEY   NUMBER,
    SUBCATEGORY_NAME  VARCHAR
);

CREATE OR REPLACE TABLE RAW.SALES (
    TRANS_ID     NUMBER,
    PROD_KEY     NUMBER,
    STORE_KEY    NUMBER,
    TRANS_DT     DATE,
    TRANS_TIME   NUMBER,
    PRIORITY     VARCHAR,
    SALES_QTY    FLOAT,
    SALES_PRICE  FLOAT,
    SALES_AMT    FLOAT,
    DISCOUNT     FLOAT,
    SALES_COST   FLOAT,
    SALES_MGRN   FLOAT,
    SHIPMODE     VARCHAR,
    SHIP_COST    FLOAT
);

SHOW TABLES IN SCHEMA RAW;

Declare the types deliberately rather than letting a load wizard infer them — `TRANS_DT` must be a real `DATE`, because the date dimension and every time-based metric depend on it.

### Question 5

Create a named **file format** for the CSVs, so the parsing rules are one
object every `COPY` reuses instead of being retyped.

In [ ]:
USE DATABASE DEMO_DB;

CREATE OR REPLACE FILE FORMAT RAW.CSV_FF
    TYPE = CSV
    SKIP_HEADER = 1                        -- both files have a header row
    FIELD_DELIMITER = ','
    FIELD_OPTIONALLY_ENCLOSED_BY = '"'     -- text values are quoted
    DATE_FORMAT = 'MM/DD/YYYY'             -- TRANS_DT is written M/D/YYYY
    NULL_IF = ('', 'NULL')
    EMPTY_FIELD_AS_NULL = TRUE;

DESC FILE FORMAT RAW.CSV_FF;

`DATE_FORMAT` is the one that bites. `TRANS_DT` is `M/D/YYYY` (e.g. `3/7/2010`); without it, every date whose day and month are both ≤ 12 is silently misread rather than rejected — a wrong answer instead of an error. Naming the format means the two `COPY` statements cannot drift apart.

### Question 6

Create the stages: one for the CSVs, one to hold the dbt project.

In [ ]:
USE DATABASE DEMO_DB;

-- The CSVs you upload.
CREATE STAGE IF NOT EXISTS RAW.LOAD_STAGE
    DIRECTORY = (ENABLE = TRUE)
    FILE_FORMAT = RAW.CSV_FF;        -- the named format from the last question

-- The dbt project files this notebook writes and uploads.
CREATE STAGE IF NOT EXISTS RAW.DBT_PROJECT_STAGE
    DIRECTORY = (ENABLE = TRUE)
    ENCRYPTION = (TYPE = 'SNOWFLAKE_SSE');

SHOW STAGES IN DATABASE DEMO_DB;

`DATE_FORMAT = 'MM/DD/YYYY'` is the important one. `TRANS_DT` is written `M/D/YYYY` (e.g. `3/7/2010`); without this, every date whose day and month are both ≤ 12 is silently misread rather than rejected.

### Question 7

Upload `products.csv` and `sales.csv` to `RAW.LOAD_STAGE`, then load them.

Upload first — in Snowsight: **Data → Databases → DEMO_DB → RAW → Stages →
LOAD_STAGE → + Files**. Then run the `COPY INTO`.

In [ ]:
USE DATABASE DEMO_DB;

-- Confirm the upload landed before copying.
LIST @RAW.LOAD_STAGE;

COPY INTO RAW.PRODUCT FROM @RAW.LOAD_STAGE/products.csv
    FILE_FORMAT = RAW.CSV_FF
    ON_ERROR = 'ABORT_STATEMENT';

COPY INTO RAW.SALES FROM @RAW.LOAD_STAGE/sales.csv
    FILE_FORMAT = RAW.CSV_FF
    ON_ERROR = 'ABORT_STATEMENT';

`LIST` first: if the filenames differ from what `COPY` expects (a client that compresses on upload would add `.gz`), fix the path rather than guessing. `ON_ERROR = 'ABORT_STATEMENT'` makes a bad row fail loudly instead of being skipped.

### Question 8

Verify the load: row counts, and the date range that proves the date format
was applied.

In [ ]:
SELECT 'product' AS table_name, COUNT(*) AS row_count FROM DEMO_DB.RAW.PRODUCT
UNION ALL
SELECT 'sales', COUNT(*) FROM DEMO_DB.RAW.SALES;

-- Must span 2009-01-01 to 2012-12-30. NULLs or a wrong range mean the date
-- format did not apply -- TRUNCATE and load again.
SELECT MIN(TRANS_DT) AS first_day,
       MAX(TRANS_DT) AS last_day,
       COUNT(DISTINCT TRANS_DT) AS distinct_days
FROM DEMO_DB.RAW.SALES;

Expect **1,214** products and **100,000** sales. A row count only proves arrival; the date range is what proves the parse. Checking both is the habit worth forming.

## PART B — writing the dbt project

Now the transformation layer. You assemble a complete dbt project as files, then
hand it to Snowflake to run.

The cells write to `/tmp/demo` inside the notebook, using plain `pathlib` — no
shell and no magics, both of which a Snowflake notebook may not give you.

### Question 9

Create the project skeleton — one folder per layer, plus macros, seeds,
snapshots and analyses.

In [ ]:
for d in ["models/staging", "models/edw", "models/marts", "models/semantic",
          "macros", "seeds", "snapshots", "analyses"]:
    (Path(PROJ) / d).mkdir(parents=True, exist_ok=True)

print("skeleton under", PROJ)
for p in sorted(Path(PROJ).rglob("*")):
    print("  ", p.relative_to(PROJ))

Each model folder becomes a layer *and* a schema. That mapping is what makes the lineage readable and lets you build one layer at a time.

### Question 10

Write `dbt_project.yml` — the file that makes a directory a dbt project. It
routes each folder to its schema and sets the project vars.

In [ ]:
p = Path(PROJ) / "dbt_project.yml"
p.parent.mkdir(parents=True, exist_ok=True)
p.write_text("""# demo — the dbt project this lab builds.
#
# Every dbt project needs this file; it is how dbt knows a directory is a
# project. It names the project, points at a profile in profiles.yml, and sets
# project-level defaults for each folder.

name: 'demo'
version: '1.0.0'
config-version: 2

profile: 'demo'

model-paths: ["models"]
snapshot-paths: ["snapshots"]
macro-paths: ["macros"]
seed-paths: ["seeds"]
test-paths: ["tests"]
analysis-paths: ["analyses"]

target-path: "target"
clean-targets:
  - "target"
  - "dbt_packages"

# vars: read in a model by calling var('name', default) in Jinja.
#   dbt_date:time_zone   dbt-expectations' date macros need a timezone.
#   heavy_wgt_threshold  used by the unit-tested weight_class logic.
vars:
  "dbt_date:time_zone": "America/Los_Angeles"
  heavy_wgt_threshold: 20

# ONE FOLDER PER LAYER — the analytics-engineering convention, and the lineage
# you trace in Part H: raw -> staging -> edw -> marts.
#
#   staging/  -> STG    thin, 1:1 with the raw sources. Views: no storage, and
#                       downstream models always see the freshest rows.
#   edw/      -> EDW    conformed dimensions and facts (the star schema).
#                       Tables: end users and BI query these constantly.
#   marts/    -> MARTS  business-facing reports. Tables, for speed.
#   seeds     -> SEED   version-controlled reference data.
#
# The custom generate_schema_name macro (macros/generate_schema_name.sql) makes
# these names literal; dbt's default would prefix them with the target schema.
models:
  demo:
    staging:
      +materialized: view
      +schema: stg
    edw:
      +materialized: table
      +schema: edw
    marts:
      +materialized: table
      +schema: marts
    # The semantic folder holds the MetricFlow time spine (a dense calendar).
    # It is warehouse infrastructure the metrics sit on, so it lands in EDW —
    # without this it would fall back to the profile's default schema.
    semantic:
      +materialized: table
      +schema: edw

seeds:
  demo:
    +schema: seed

snapshots:
  demo:
    +schema: stg
""")
print("wrote", p)

`staging → STG` (views: no storage, always fresh), `edw` and `marts` → tables (queried constantly). The `profile:` key names the profile in the next question.

### Question 11

Write `profiles.yml`. Running inside Snowflake, it carries **no credentials** —
the executing role is the identity.

In [ ]:
p = Path(PROJ) / "profiles.yml"
p.parent.mkdir(parents=True, exist_ok=True)
p.write_text("""# profiles.yml for DBT PROJECTS ON SNOWFLAKE.
#
# Use this version when the project runs INSIDE Snowflake as a native DBT
# PROJECT object (rename it to profiles.yml in the uploaded project). Snowflake
# authenticates with the ROLE that executes the project, so there is no account,
# user, password or key here at all — the credential problem disappears.
#
# Compare with:
#   profiles.example.yml  the local/CLI version, which does need connection details
#   the notebook's copy    the local version, reading SNOWFLAKE_* from env vars
#
# `database`, `schema`, `warehouse` and `role` are still declared: they set the
# defaults the models build into. The custom generate_schema_name macro then
# routes each folder to STG / EDW / MARTS / SEED as usual.

demo:
  target: dev
  outputs:
    dev:
      type: snowflake
      # No account/user/password: the executing role supplies the identity.
      database: DEMO_DB
      schema: dev          # default only; the layers override it per folder
      warehouse: COMPUTE_WH
      role: SYSADMIN
      threads: 4
""")
print("wrote", p)

Compare with a laptop profile, which needs account, user and password. Here `database`, `schema`, `warehouse` and `role` only set the *defaults* models build into; the custom schema macro overrides them per folder. Nothing secret is ever written.

### Question 12

Write `packages.yml` — dbt_utils (surrogate keys) and dbt-expectations (the
Great-Expectations-style tests).

In [ ]:
p = Path(PROJ) / "packages.yml"
p.parent.mkdir(parents=True, exist_ok=True)
p.write_text("""# PACKAGES — pre-built, shareable dbt modules from the dbt Hub. Adding one gives
# you its macros, tests and sometimes models. Install with `dbt deps`, which
# downloads them into dbt_packages/.

packages:
  # Utility macros. This lab uses generate_surrogate_key for the star schema's
  # keys; it also has date spines, cross-database shims and extra tests.
  - package: dbt-labs/dbt_utils
    version: 1.1.1

  # The dbt port of Great Expectations: range, distribution, shape and freshness
  # assertions beyond dbt's four generic tests. Used in models/marts.
  #
  # `dbt deps` prints a deprecation notice for this one — calogica/dbt_expectations
  # has moved to metaplane/dbt_expectations. The version below is what the lab and
  # the lecture specify and it still installs and runs, so the notice is expected,
  # not an error. On a real project, switch to metaplane/dbt_expectations.
  - package: calogica/dbt_expectations
    version: [">=0.9.0", "<0.10.0"]
""")
print("wrote", p)

These install with `dbt deps`. Note the network caveat in Part D: code inside Snowflake has no outbound internet by default.

### Question 13

Write the custom `generate_schema_name` macro so the layer schemas are used
verbatim.

In [ ]:
p = Path(PROJ) / "macros/generate_schema_name.sql"
p.parent.mkdir(parents=True, exist_ok=True)
p.write_text("""-- Custom generate_schema_name — the macro that makes the layer schemas literal.
--
-- dbt's DEFAULT version PREFIXES a custom schema with the target schema, so a
-- model configured schema='edw' would build in "<target.schema>_edw". That
-- default exists so several developers sharing one warehouse do not overwrite
-- each other. This lab wants exactly STG / EDW / MARTS / SEED, so we override it.
--
-- The .sql file name does not need to match the macro name.

{% macro generate_schema_name(custom_schema_name, node) -%}

    {%- set default_schema = target.schema -%}
    {%- if custom_schema_name is none -%}

        {{ default_schema }}

    {%- else -%}

        {{ custom_schema_name | trim }}

    {%- endif -%}

{%- endmacro %}
""")
print("wrote", p)

Without it, `schema: edw` builds in `<target_schema>_edw`. dbt prefixes by default so developers sharing a warehouse do not collide; this lab wants exactly `STG` / `EDW` / `MARTS`.

### Question 14

Declare the sources — the raw tables you loaded in Part A.

In [ ]:
p = Path(PROJ) / "models/staging/sources.yml"
p.parent.mkdir(parents=True, exist_ok=True)
p.write_text("""# SOURCES — the raw tables your Extract/Load step landed in Snowflake (you
# created and loaded them with the snowflake-console/ scripts).
#
# Declaring them lets models select from them by calling source('stg','product')
# in Jinja, puts them on the lineage graph, and lets you test and freshness-check
# the inputs. The source is NAMED 'stg' but the tables live in the RAW schema, so
# `schema: raw` is required — by default dbt looks for a schema matching the name.

version: 2

sources:
  - name: stg
    database: "{{ target.database }}"
    schema: raw
    description: "Raw landing zone, loaded from the lab's two CSVs."
    tables:
      - name: product
        description: "Product master, one row per product (mutable — it changes over time)."
      - name: sales
        description: "Sales transactions, one row per line item."
""")
print("wrote", p)

The source is *named* `stg` but `schema: raw` points it at the real tables. Declaring sources puts them on the lineage graph and makes them testable.

### Question 15

Write the two staging models: `stg_product_incr` (incremental, `merge`
strategy, stamping a `start_date`) and `stg_sales`.

In [ ]:
p = Path(PROJ) / "models/staging/stg_product_incr.sql"
p.parent.mkdir(parents=True, exist_ok=True)
p.write_text("""-- STAGING — 1:1 with raw.product, captured INCREMENTALLY with the merge strategy.
--
-- Materialization: incremental + incremental_strategy='merge'. On the first run
-- dbt builds the whole table; on later runs is_incremental() is true, so only
-- rows at/after the current max(start_date) are read and merged on the unique
-- key. That is the pattern for large tables where a full rebuild is too slow.
--
-- Builds in STG (see dbt_project.yml).

{{
    config(
        materialized = 'incremental',
        incremental_strategy = 'merge',
        unique_key = ['prod_key', 'prod_name', 'vol', 'wgt', 'brand_name', 'status_code', 'status_code_name', 'category_key', 'category_name', 'subcategory_key', 'subcategory_name']
    )
}}

{% if is_incremental() %}

{% set MAX_START_DATE_query %}
select ifnull(max(start_date), '1900-01-01') from {{ this }} as MAX_START_DT
{% endset %}

{% if execute %}
{% set MAX_START_DT = run_query(MAX_START_DATE_query).columns[0][0] %}
{% endif %}

{% endif %}

select
    prod_key,
    prod_name,
    vol,
    wgt,
    brand_name,
    status_code,
    status_code_name,
    category_key,
    category_name,
    subcategory_key,
    subcategory_name,
    sysdate() as start_date
from
    {{ source('stg', 'product') }}
{% if is_incremental() %}
where start_date >= '{{ MAX_START_DT }}'
{% endif %}
""")
print("wrote", p)

Every incremental model has three parts: the `config`, the `is_incremental()` block holding the cutoff, and the filter itself. First run builds everything; later runs read only rows past `max(start_date)` and merge them.

### Question 16

Write `stg_sales` — rename and select only, no joins, no aggregation.

In [ ]:
p = Path(PROJ) / "models/staging/stg_sales.sql"
p.parent.mkdir(parents=True, exist_ok=True)
p.write_text("""-- STAGING — 1:1 with raw.sales. Rename, cast, select. NOTHING else.
--
-- Staging models are deliberately boring: same grain as the source, no joins and
-- no aggregation (those belong in edw/). Materialized as a view, so it costs no
-- storage and always reflects the latest raw rows.

select
    trans_id,
    trans_dt   as cal_dt,
    store_key,
    prod_key,
    priority,
    sales_qty,
    sales_price,
    sales_amt,
    sales_cost,
    sales_mgrn,
    discount,
    shipmode   as ship_mode,
    ship_cost
from {{ source('stg', 'sales') }}
""")
print("wrote", p)

Deliberately boring, and that is the discipline: staging stays 1:1 with the source so every downstream model has one clean thing to build on.

### Question 17

Document the staging layer — a reusable doc block, then the descriptions and
tests that consume it.

In [ ]:
p = Path(PROJ) / "models/staging/_stg__docs.md"
p.parent.mkdir(parents=True, exist_ok=True)
p.write_text("""{% docs prod_key_doc %}
The **product key** — the natural identifier of a product in the source system.

It is unique per product *at a point in time*, but repeats across historical
versions in the Type 6 dimension. That is exactly why `dim_product_t6` is keyed
by a surrogate key over `(prod_key, valid_from)` rather than by `prod_key`.

Any column can reuse this text by calling the doc function with this block's
name, `prod_key_doc`, from its `description:`.
{% enddocs %}

{% docs surrogate_key_doc %}
A **surrogate key**: a warehouse-generated identifier with no business meaning,
built by hashing the natural key plus the version's validity start. It gives the
fact table a single stable column to join on, and stays correct when the natural
key gains new versions.
{% enddocs %}
""")
print("wrote", p)

A doc block is markdown any description can reuse by calling `doc('prod_key_doc')`. Write the definition once, use it on every model that has the column.

### Question 18

Write the staging model properties.

In [ ]:
p = Path(PROJ) / "models/staging/stg__models.yml"
p.parent.mkdir(parents=True, exist_ok=True)
p.write_text("""# Documentation and tests for the staging layer.
#
# Descriptions live beside the models they describe. The prod_key description
# reuses the doc block in _stg__docs.md via the doc() function.
#
# NOTE: dbt renders .yml files as Jinja BEFORE parsing them, and that includes
# '#' comment lines — so a live tag written in a comment here would actually be
# executed. Describe Jinja in words in comments; use it for real in values.

version: 2

models:
  - name: stg_product_incr
    description: "Staging of raw.product, captured incrementally with a start_date stamp."
    columns:
      - name: prod_key
        description: '{{ doc("prod_key_doc") }}'
        tests:
          - not_null
      - name: start_date
        description: "When this version of the product row was captured."
        tests:
          - not_null

  - name: stg_sales
    description: "Staging of raw.sales — renamed and typed, one row per transaction line."
    columns:
      - name: trans_id
        description: "Transaction identifier from the source. NOT unique on its own."
        tests:
          - not_null
      - name: prod_key
        description: '{{ doc("prod_key_doc") }}'
        tests:
          - not_null
      - name: cal_dt
        description: "Calendar date of the transaction (renamed from trans_dt)."
        tests:
          - not_null
""")
print("wrote", p)

Configuration sits next to what it describes: `dbt_project.yml` for project-wide defaults, folder YAML for the detail.

## PART C — seeds, snapshots, and the Type 6 dimension

**Seeds** are reference data in the repo. **Snapshots** capture how a mutable
source changes. Together they feed a **Type 6** slowly-changing dimension and the
star schema around it.

### Question 19

The business reports by region, but sales only has a `store_key`. Write the
`store_master` seed to bridge that gap.

In [ ]:
p = Path(PROJ) / "seeds/store_master.csv"
p.parent.mkdir(parents=True, exist_ok=True)
p.write_text("""store_key,store_name,region,country
8106,Store-8106,East,CA
8107,Store-8107,East,CA
9001,Store-9001,West,US
9002,Store-9002,West,US
9003,Store-9003,West,US
9004,Store-9004,Central,US
9005,Store-9005,Central,US
9006,Store-9006,Central,US
9007,Store-9007,North,CA
9008,Store-9008,North,CA
9011,Store-9011,South,US
9012,Store-9012,South,US
9013,Store-9013,South,US
""")
print("wrote", p)

13 stores, matching every `store_key` in the data. Because it lives in the project, a region change is a reviewable diff instead of a silent spreadsheet edit.

### Question 20

Finance measures actuals against plan, and the targets exist only in a
spreadsheet. Write the `category_targets` seed, and the seed properties.

In [ ]:
p = Path(PROJ) / "seeds/category_targets.csv"
p.parent.mkdir(parents=True, exist_ok=True)
p.write_text("""category_name,annual_sales_target
category-1,30000000
category-2,25000000
category-3,20000000
category-4,15000000
category-5,10000000
""")
print("wrote", p)

A planning number no source system holds — exactly what seeds are for. It drives the actuals-vs-target mart in Part D.

### Question 21

Write `seeds.yml` — descriptions and tests for both seeds.

In [ ]:
p = Path(PROJ) / "seeds/seeds.yml"
p.parent.mkdir(parents=True, exist_ok=True)
p.write_text("""# Seeds are reference data that lives in the repo and is loaded by `dbt seed`.
# They exist to answer a BUSINESS REQUIREMENT that the source system cannot:
#
#   store_master      the source only has a store_key. The business reports by
#                     REGION, so the key->region map lives here, in version
#                     control, where a change is a reviewable diff.
#   category_targets  annual sales target per category — a planning number that
#                     exists in a spreadsheet, not in any source table. The
#                     marts layer measures actuals against it.
#
# Seeds are referenced with ref(), exactly like models.

version: 2

seeds:
  - name: store_master
    description: "Store key -> name, region, country. Drives all regional reporting."
    columns:
      - name: store_key
        description: "Natural key of the store, as it appears in raw.sales."
        tests:
          - unique
          - not_null
      - name: region
        tests:
          - not_null
          - accepted_values:
              arguments:
                values: ['East', 'West', 'Central', 'North', 'South']

  - name: category_targets
    description: "Annual sales target per product category (planning input)."
    columns:
      - name: category_name
        tests:
          - unique
          - not_null
""")
print("wrote", p)

Seeds get tested like models. `store_key` unique and not-null matters here: a duplicate would fan out the fact table when the dimension joins.

### Question 22

Write the product snapshot, using the **check** strategy.

In [ ]:
p = Path(PROJ) / "snapshots/product_snapshot.sql"
p.parent.mkdir(parents=True, exist_ok=True)
p.write_text("""-- SNAPSHOT — dbt's built-in Type 2 history capture, in the classic SQL block.
--
-- raw.product is MUTABLE: a category or a name can change in place, and the old
-- value is lost. A snapshot records each state so history survives. It is the
-- foundation the Type 6 dimension is built on.
--
-- strategy='check' with check_cols='all': raw.product has no reliable
-- "updated_at" column, so dbt compares every column per prod_key. When anything
-- differs it closes the old row (dbt_valid_to = now) and inserts a new one
-- (dbt_valid_from = now, dbt_valid_to = null).
--
-- Built by `dbt snapshot` (or `dbt build`) — NOT by `dbt run`.

{% snapshot product_snapshot %}

{{
    config(
        target_schema='stg',
        strategy='check',
        unique_key='prod_key',
        check_cols='all',
    )
}}

select
    *
from
    {{ source('stg', 'product') }}
order by
    prod_key

{% endsnapshot %}
""")
print("wrote", p)

`raw.product` is mutable — a category changes in place and the old value is gone. `check` with `check_cols='all'` compares every column and, on any change, closes the old row and inserts a new one. That is the Type 2 history the Type 6 dimension is built from.

### Question 23

Write the sales snapshot using the **timestamp** strategy, in the modern YAML
form.

In [ ]:
p = Path(PROJ) / "snapshots/sales_snapshot.yml"
p.parent.mkdir(parents=True, exist_ok=True)
p.write_text("""# The MODERN, YAML-based snapshot configuration (dbt >= 1.9), recommended over
# the SQL snapshot block for new snapshots — compare with product_snapshot.sql.
#
# This one uses the TIMESTAMP strategy, which is preferred whenever the source
# has a trustworthy "last updated" column: dbt trusts trans_dt instead of
# comparing every column, which is cheaper and records the real change time.
# Use the check strategy (as product_snapshot does) only when no such column
# exists.

snapshots:
  - name: sales_snapshot
    relation: source('stg', 'sales')
    description: "Timestamp-strategy snapshot of raw.sales, keyed on trans_id."
    config:
      schema: stg
      unique_key: trans_id
      strategy: timestamp
      updated_at: trans_dt
""")
print("wrote", p)

Two strategies, side by side: timestamp is preferred whenever a trustworthy updated-at column exists (cheaper, and it records the real change time); check is the fallback when none does. YAML is the recommended form for new snapshots.

### Question 24

Write the `to_active_flag` macro — the "null close date means current" rule,
written once.

In [ ]:
p = Path(PROJ) / "macros/to_active_flag.sql"
p.parent.mkdir(parents=True, exist_ok=True)
p.write_text("""-- A custom macro. Macros are Jinja functions that RETURN SQL text, which dbt
-- pastes into the model at compile time — the direct analogue of a function in
-- Python. This one wraps the "a null close date means the row is current" rule
-- so it is written once and reused, not copy-pasted into every dimension.
--
-- Usage in a model:   call to_active_flag('valid_to') in curly braces,
--                     aliased as is_current
-- Compiles to:        iff(valid_to is null, true, false) as is_current
--
-- (The usage line is spelled out rather than written with real Jinja
--  delimiters: dbt renders this file as a template before parsing it, so a live
--  tag inside a comment would actually be executed.)

{% macro to_active_flag(valid_to_column) -%}
    iff({{ valid_to_column }} is null, true, false)
{%- endmacro %}
""")
print("wrote", p)

Macros are Jinja functions returning SQL. Change the rule here and every dimension that uses it follows.

### Question 25

Write `dim_product_t6` — the **Type 6** dimension, keyed by a surrogate key.

In [ ]:
p = Path(PROJ) / "models/edw/dim_product_t6.sql"
p.parent.mkdir(parents=True, exist_ok=True)
p.write_text("""-- EDW — the product dimension as a TYPE 6 slowly-changing dimension.
--
-- Type 6 is the hybrid "1 + 2 + 3" (1 x 2 x 3 = 6, which is where the name comes
-- from). Each row of this table carries all three views of an attribute:
--
--   TYPE 2  category_name           the value AS OF that version. One row per
--                                   version, bounded by valid_from/valid_to.
--                                   Answers "what was it at the time?"
--   TYPE 1  current_category_name   the CURRENT value, repeated on every
--                                   historical row of the same product.
--                                   Answers "show me all history re-stated
--                                   under today's category."
--   TYPE 3  previous_category_name  the value from the version immediately
--                                   before this one. Answers "what did it
--                                   change from?" without a self-join.
--
-- That combination is what makes Type 6 useful: the same fact can be reported
-- "as was" (join on the Type 2 column) or "as is" (join on the Type 1 column)
-- with no change to the fact table.
--
-- History comes from the snapshot, so this model is pure SQL over dbt's
-- dbt_valid_from / dbt_valid_to. Materialized as a table (edw default).

with versions as (

    select
        prod_key,
        prod_name,
        brand_name,
        category_name,
        subcategory_name,
        status_code_name,
        dbt_valid_from as valid_from,
        dbt_valid_to   as valid_to
    from {{ ref('product_snapshot') }}

),

-- TYPE 1: the current value per product, to be stamped onto every row.
current_values as (

    select
        prod_key,
        category_name    as current_category_name,
        prod_name        as current_prod_name,
        subcategory_name as current_subcategory_name
    from versions
    where valid_to is null

),

-- TYPE 3: the value from the previous version, via lag over the version order.
with_previous as (

    select
        v.*,
        lag(v.category_name) over (
            partition by v.prod_key order by v.valid_from
        ) as previous_category_name
    from versions v

)

select
    -- The surrogate key: unique per VERSION, not per product. This is the column
    -- the fact table joins on.
    {{ dbt_utils.generate_surrogate_key(['w.prod_key', 'w.valid_from']) }} as product_sk,

    w.prod_key,                                 -- natural key

    -- Type 2 — as of this version
    w.prod_name,
    w.brand_name,
    w.category_name,
    w.subcategory_name,
    w.status_code_name,

    -- Type 1 — today's value, on every row
    c.current_prod_name,
    c.current_category_name,
    c.current_subcategory_name,

    -- Type 3 — the value it changed from
    w.previous_category_name,

    -- validity window
    w.valid_from,
    w.valid_to,
    {{ to_active_flag('w.valid_to') }} as is_current

from with_previous w
left join current_values c
    on w.prod_key = c.prod_key
""")
print("wrote", p)

Type 6 = 1 + 2 + 3 (1×2×3=6). Three CTEs, one per view: `versions` (Type 2, as-of), `current_values` (Type 1, today's value on every row), `with_previous` (Type 3, via `lag`). The surrogate key hashes `(prod_key, valid_from)`, so each *version* is uniquely addressable — which is exactly why `prod_key` alone cannot key an SCD.

### Question 26

Write `dim_store` (from the seed) and `dim_date`.

In [ ]:
p = Path(PROJ) / "models/edw/dim_store.sql"
p.parent.mkdir(parents=True, exist_ok=True)
p.write_text("""-- EDW — the store dimension, built entirely from a SEED.
--
-- The source system only ever gives us a store_key. Region is a business
-- attribute that lives in version control (seeds/store_master.csv), so the
-- dimension is the seed plus a surrogate key. A Type 1 dimension: no history,
-- the current row is the only row.

select
    {{ dbt_utils.generate_surrogate_key(['store_key']) }} as store_sk,
    store_key,
    store_name,
    region,
    country
from {{ ref('store_master') }}
""")
print("wrote", p)

A Type 1 dimension: a store's region is corrected, not versioned. The seed plus a surrogate key *is* the dimension.

### Question 27

Write `dim_date`, derived from the dates present in sales.

In [ ]:
p = Path(PROJ) / "models/edw/dim_date.sql"
p.parent.mkdir(parents=True, exist_ok=True)
p.write_text("""-- EDW — the date dimension, derived from the dates actually present in sales.
--
-- Uses a smart integer key (YYYYMMDD) rather than a hash: date keys are the one
-- place a readable surrogate key is conventional, because it sorts and ranges
-- naturally. Every star schema has one of these; it is what lets the business
-- ask for "last quarter" without writing date arithmetic in every query.

with dates as (

    select distinct cal_dt
    from {{ ref('stg_sales') }}
    where cal_dt is not null

)

select
    to_number(to_char(cal_dt, 'YYYYMMDD'))  as date_key,
    cal_dt                                   as full_date,
    year(cal_dt)                             as year_num,
    quarter(cal_dt)                          as quarter_num,
    month(cal_dt)                            as month_num,
    monthname(cal_dt)                        as month_name,
    day(cal_dt)                              as day_of_month,
    dayofweek(cal_dt)                        as day_of_week,
    dayname(cal_dt)                          as day_name,
    iff(dayofweek(cal_dt) in (0, 6), true, false) as is_weekend
from dates
""")
print("wrote", p)

It uses a **smart key** (`YYYYMMDD` as an integer) — the one place a readable surrogate key is conventional, because it sorts and ranges naturally.

### Question 28

Write `fct_sales` — the fact at the centre of the star, incremental with the
**delete+insert** strategy.

In [ ]:
p = Path(PROJ) / "models/edw/fct_sales.sql"
p.parent.mkdir(parents=True, exist_ok=True)
p.write_text("""-- EDW — the FACT table at the centre of the star.
--
--                    dim_date
--                        |
--     dim_product_t6 — fct_sales — dim_store
--
-- GRAIN: one row per (date, product version, store). Every fact table must have
-- a stated grain, and every measure must be additive at that grain.
--
-- The fact carries only SURROGATE KEYS and MEASURES — no descriptive attributes.
-- That is what makes it a star: descriptions live in the dimensions, so a
-- category rename does not require touching the fact.
--
-- It joins dim_product_t6 on is_current, so each sale attaches to the product's
-- CURRENT version. (Joining on the validity window instead —
-- cal_dt between valid_from and valid_to — would give you "as was" reporting;
-- with Type 6 you get that anyway from the Type 2 columns.)
--
-- Materialization: incremental with the DELETE+INSERT strategy. For a recomputed
-- aggregate, an updated day's rows must be REPLACED, not merged column by
-- column — delete+insert deletes the matching unique_key rows and re-inserts
-- them. On a later run only dates at/after the current max are reprocessed.

{{
    config(
        materialized = 'incremental',
        incremental_strategy = 'delete+insert',
        unique_key = ['date_key', 'product_sk', 'store_sk']
    )
}}

{% if is_incremental() %}

{% set MAX_CAL_DATE_query %}
select ifnull(max(full_date), '1900-01-01') from {{ this }} as MAX_CAL_DT
{% endset %}

{% if execute %}
{% set MAX_CAL_DT = run_query(MAX_CAL_DATE_query).columns[0][0] %}
{% endif %}

{% endif %}

select
    -- foreign keys into the dimensions
    to_number(to_char(s.cal_dt, 'YYYYMMDD')) as date_key,
    p.product_sk,
    st.store_sk,

    -- kept for readability / the incremental filter
    s.cal_dt as full_date,

    -- additive measures at this grain
    sum(s.sales_qty)   as sales_qty,
    sum(s.sales_amt)   as sales_amt,
    sum(s.sales_cost)  as sales_cost,
    sum(s.sales_mgrn)  as sales_mgrn,
    sum(s.ship_cost)   as ship_cost,

    -- non-additive: an average cannot be summed further, so it is flagged here
    avg(s.sales_price) as avg_sales_price,
    avg(s.discount)    as avg_discount,

    current_timestamp() as dbt_loaded_at

from {{ ref('stg_sales') }} s
left join {{ ref('dim_product_t6') }} p
    on s.prod_key = p.prod_key
   and p.is_current
left join {{ ref('dim_store') }} st
    on s.store_key = st.store_key
{% if is_incremental() %}
where s.cal_dt >= '{{ MAX_CAL_DT }}'
{% endif %}
group by 1, 2, 3, 4
""")
print("wrote", p)

**Grain:** one row per (date, product version, store), stated explicitly because every measure must be additive at the grain. The fact holds only keys and measures — descriptions live in the dimensions. delete+insert, not merge, because a recomputed aggregate must be *replaced* rather than merged column by column.

### Question 29

Write the EDW properties: documentation plus all four generic tests, including
referential integrity from fact to each dimension.

In [ ]:
p = Path(PROJ) / "models/edw/edw__models.yml"
p.parent.mkdir(parents=True, exist_ok=True)
p.write_text("""# Documentation and DATA TESTS for the EDW star schema.
#
# All four generic tests dbt ships with appear here:
#   unique / not_null      column-level integrity
#   accepted_values        a column's values are from a known set
#   relationships          referential integrity — every fact FK finds its dim
#
# A generic test's parameters go under `arguments:` in current dbt. Older
# material (including the lecture slides) puts them at the top level; that still
# runs but dbt now warns it is deprecated.

version: 2

models:
  - name: dim_product_t6
    description: >
      Type 6 product dimension. One row per product VERSION, carrying the Type 2
      value as of that version, the Type 1 current value, and the Type 3
      previous value.
    columns:
      - name: product_sk
        description: '{{ doc("surrogate_key_doc") }}'
        tests:
          # This is the point of the surrogate key: prod_key is NOT unique here
          # (a product has one row per version), but product_sk is.
          - unique
          - not_null
      - name: prod_key
        description: '{{ doc("prod_key_doc") }}'
        tests:
          - not_null
      - name: is_current
        description: "True on the one row per product whose valid_to is null."
        tests:
          - not_null
      - name: status_code_name
        tests:
          - accepted_values:
              arguments:
                values: ['active']

  - name: dim_store
    description: "Type 1 store dimension, built from the store_master seed."
    columns:
      - name: store_sk
        tests: [unique, not_null]
      - name: store_key
        tests: [unique, not_null]
      - name: region
        tests:
          - accepted_values:
              arguments:
                values: ['East', 'West', 'Central', 'North', 'South']

  - name: dim_date
    description: "Date dimension derived from the dates present in sales."
    columns:
      - name: date_key
        description: "Smart integer key, YYYYMMDD."
        tests: [unique, not_null]
      - name: full_date
        tests: [unique, not_null]

  - name: fct_sales
    description: "Sales fact. Grain: one row per (date, product version, store)."
    columns:
      - name: date_key
        tests:
          - not_null
          # Referential integrity: every date_key must exist in dim_date.
          - relationships:
              arguments:
                to: ref('dim_date')
                field: date_key
      - name: product_sk
        tests:
          - not_null
          - relationships:
              arguments:
                to: ref('dim_product_t6')
                field: product_sk
      - name: store_sk
        tests:
          - not_null
          - relationships:
              arguments:
                to: ref('dim_store')
                field: store_sk
""")
print("wrote", p)

`unique` on `product_sk` is the surrogate key paying off — `prod_key` repeats across versions and would fail. The `relationships` tests are the star schema's integrity guarantee: every fact FK must find its dimension row.

### Question 30

Write the **unit tests** — automated scenarios for the Type 6 logic.

In [ ]:
p = Path(PROJ) / "models/edw/unit_tests.yml"
p.parent.mkdir(parents=True, exist_ok=True)
p.write_text("""# UNIT TESTS — automated test scenarios for the Type 6 SCD logic.
#
# A DATA test runs against whatever is in the warehouse. A UNIT test (dbt >= 1.8)
# feeds the model MOCK rows and asserts the exact output, so it checks the LOGIC
# with no real data, deterministically, in seconds. That is what you want for
# SCD rules, which are otherwise only exercised when a source happens to change.
#
# The scenario below is a product with THREE versions: it starts in category-1,
# moves to category-2, then to category-3 (the current one). That single fixture
# pins down all three Type 6 behaviours at once.

version: 2

unit_tests:
  - name: type6_scd_tracks_all_three_views
    description: >
      Given three versions of one product, dim_product_t6 must emit three rows
      carrying the Type 2 (as-of), Type 1 (current, repeated) and Type 3
      (previous) values, with only the newest row flagged is_current.
    model: dim_product_t6
    given:
      - input: ref('product_snapshot')
        rows:
          - {prod_key: 1, prod_name: 'Widget', brand_name: 'b1', category_name: 'category-1', subcategory_name: 's1', status_code_name: 'active', dbt_valid_from: '2024-01-01', dbt_valid_to: '2024-06-01'}
          - {prod_key: 1, prod_name: 'Widget', brand_name: 'b1', category_name: 'category-2', subcategory_name: 's1', status_code_name: 'active', dbt_valid_from: '2024-06-01', dbt_valid_to: '2024-09-01'}
          - {prod_key: 1, prod_name: 'Widget', brand_name: 'b1', category_name: 'category-3', subcategory_name: 's1', status_code_name: 'active', dbt_valid_from: '2024-09-01', dbt_valid_to: null}
    expect:
      rows:
        # Type 2 = category_name (differs per row)
        # Type 1 = current_category_name (category-3 on EVERY row)
        # Type 3 = previous_category_name (null on the first version)
        - {prod_key: 1, category_name: 'category-1', current_category_name: 'category-3', previous_category_name: null,        is_current: false}
        - {prod_key: 1, category_name: 'category-2', current_category_name: 'category-3', previous_category_name: 'category-1', is_current: false}
        - {prod_key: 1, category_name: 'category-3', current_category_name: 'category-3', previous_category_name: 'category-2', is_current: true}

  - name: date_dimension_flags_weekends
    description: "dim_date's is_weekend flag is right on the boundary days."
    model: dim_date
    given:
      - input: ref('stg_sales')
        rows:
          - {cal_dt: '2024-01-05'}   # Friday
          - {cal_dt: '2024-01-06'}   # Saturday
          - {cal_dt: '2024-01-07'}   # Sunday
          - {cal_dt: '2024-01-08'}   # Monday
    expect:
      rows:
        - {date_key: 20240105, is_weekend: false}
        - {date_key: 20240106, is_weekend: true}
        - {date_key: 20240107, is_weekend: true}
        - {date_key: 20240108, is_weekend: false}
""")
print("wrote", p)

A data test checks the warehouse; a unit test checks the *logic* against mock rows, deterministically and with no real data. One fixture — a product moving category-1 → 2 → 3 — pins down all three Type 6 views at once.

## PART D — marts, quality, and the semantic layer

### Question 31

Write the pivot macro — it generates one SUM column per category with a Jinja
loop.

In [ ]:
p = Path(PROJ) / "macros/pivot_category_amounts.sql"
p.parent.mkdir(parents=True, exist_ok=True)
p.write_text("""-- A macro that GENERATES SQL with a Jinja loop — the pattern that makes dbt more
-- than string templating. Given a column and a list of categories it writes one
-- conditional SUM per category, so a five-category pivot is a few lines of Jinja
-- instead of five hand-maintained columns that drift.
--
-- The Jinja on show: a for/endfor loop, with loop.last so a comma goes between
-- items but not after the last; expression interpolation for the category and
-- the column name; the replace('-', '_') filter, because 'category-1' is not a
-- legal column name; and the dashes on the tags, which trim whitespace so the
-- compiled SQL stays readable.
--
-- NOTE: this comment spells those tags out in words on purpose. dbt renders
-- every file as a Jinja template BEFORE parsing it and does not skip SQL
-- comments — a literal loop tag written here would open a control-flow block and
-- fail with "block definition inside control flow".
--
-- Run `dbt compile --select rpt_category_pivot` to see what it expands to.

{% macro pivot_category_amounts(amount_column, categories) -%}
    {%- for c in categories %}
    sum(case when category_name = '{{ c }}' then {{ amount_column }} else 0 end)
        as {{ c | replace('-', '_') }}
    {%- if not loop.last %},{% endif %}
    {%- endfor %}
{%- endmacro %}
""")
print("wrote", p)

Jinja doing real work: a loop, `loop.last` for comma placement, and a filter to make a legal column name. Its comment describes the tags in words on purpose — dbt renders every file as a template *before* parsing and does not skip SQL comments, so a literal tag in a comment would execute and break the macro.

### Question 32

Write the three mart models: sales by region, actuals vs target, and the
macro-generated pivot.

In [ ]:
p = Path(PROJ) / "models/marts/rpt_sales_by_region.sql"
p.parent.mkdir(parents=True, exist_ok=True)
p.write_text("""-- MARTS — the business-facing report: sales by region and month.
--
-- This is a pure star-schema query: the fact joined to its dimensions by
-- surrogate key. Note it reports on current_category_name (the Type 1 column),
-- so history is re-stated under today's categories — "as is" reporting. Swap it
-- for category_name to get "as was".

select
    d.year_num,
    d.month_num,
    d.month_name,
    s.region,
    p.current_category_name as category_name,
    sum(f.sales_qty)  as sales_qty,
    sum(f.sales_amt)  as sales_amt,
    sum(f.sales_mgrn) as sales_margin
from {{ ref('fct_sales') }} f
join {{ ref('dim_date') }}        d on f.date_key   = d.date_key
join {{ ref('dim_product_t6') }}  p on f.product_sk = p.product_sk
join {{ ref('dim_store') }}       s on f.store_sk   = s.store_sk
group by 1, 2, 3, 4, 5
""")
print("wrote", p)

A pure star query — the fact joined to its dimensions by surrogate key. It reports on `current_category_name` (Type 1), so history is re-stated under today's categories; swap to `category_name` for "as was".

### Question 33

Write `rpt_category_vs_target` — the report the seed exists for.

In [ ]:
p = Path(PROJ) / "models/marts/rpt_category_vs_target.sql"
p.parent.mkdir(parents=True, exist_ok=True)
p.write_text("""-- MARTS — actuals vs plan, the business requirement the category_targets SEED
-- exists to serve.
--
-- The warehouse has no "target" anywhere: it is a planning number that lived in
-- a spreadsheet. Putting it in a seed makes it version-controlled, testable, and
-- joinable — and this model is the payoff: a report the source systems alone
-- could never produce.

with actuals as (

    select
        p.current_category_name as category_name,
        sum(f.sales_amt)        as actual_sales
    from {{ ref('fct_sales') }} f
    join {{ ref('dim_product_t6') }} p on f.product_sk = p.product_sk
    group by 1

)

select
    t.category_name,
    t.annual_sales_target,
    coalesce(a.actual_sales, 0) as actual_sales,
    round(coalesce(a.actual_sales, 0) - t.annual_sales_target, 2) as variance,
    round(100.0 * coalesce(a.actual_sales, 0) / nullif(t.annual_sales_target, 0), 1) as pct_of_target,
    iff(coalesce(a.actual_sales, 0) >= t.annual_sales_target, 'MET', 'MISSED') as target_status
from {{ ref('category_targets') }} t
left join actuals a on t.category_name = a.category_name
""")
print("wrote", p)

Actuals from the warehouse, plan from version control. This is the business requirement the source systems alone could never answer.

### Question 34

Write `rpt_category_pivot`, calling your macro.

In [ ]:
p = Path(PROJ) / "models/marts/rpt_category_pivot.sql"
p.parent.mkdir(parents=True, exist_ok=True)
p.write_text("""-- MARTS — sales per store, pivoted into one column per category by the
-- pivot_category_amounts macro. The set tag below defines the category list in
-- one place; the macro call expands it into five SUM(CASE ...) columns.
--
-- Run `dbt compile --select rpt_category_pivot` to see the generated SQL.

{% set categories = ['category-1', 'category-2', 'category-3', 'category-4', 'category-5'] %}

with sales_by_category as (

    select
        st.store_key,
        p.current_category_name as category_name,
        f.sales_amt
    from {{ ref('fct_sales') }} f
    join {{ ref('dim_product_t6') }} p on f.product_sk = p.product_sk
    join {{ ref('dim_store') }} st     on f.store_sk   = st.store_sk

)

select
    store_key,
    {{ pivot_category_amounts('sales_amt', categories) }}
from sales_by_category
group by store_key
""")
print("wrote", p)

The `set` tag lists the categories once; the macro expands them into five columns. Change the list and the report changes.

### Question 35

Write the marts properties — the **dbt-expectations** data-quality checks.

In [ ]:
p = Path(PROJ) / "models/marts/marts__models.yml"
p.parent.mkdir(parents=True, exist_ok=True)
p.write_text("""# DATA QUALITY for the marts layer, using dbt-expectations.
#
# dbt-expectations is the dbt port of Great Expectations: a library of ready-made
# assertions that go beyond dbt's four generic tests. Where `not_null` checks a
# column, these check DISTRIBUTIONS, RANGES, SHAPES and FRESHNESS — the kinds of
# defect that pass every column-level test and still make a report wrong.
#
# Install with `dbt deps` (see packages.yml); they need the dbt_date:time_zone
# var set in dbt_project.yml.

version: 2

models:
  - name: rpt_sales_by_region
    description: "Sales by region, month and (current) category — the main dashboard feed."
    tests:
      # SHAPE: the report must never come back empty.
      - dbt_expectations.expect_table_row_count_to_be_between:
          arguments:
            min_value: 1
      # FRESHNESS: every region should have recent data.
      - dbt_expectations.expect_grouped_row_values_to_have_recent_data:
          arguments:
            group_by: [region]
            timestamp_column: month_num
            datepart: day
            interval: 400
    columns:
      - name: region
        tests:
          - not_null
          - dbt_expectations.expect_column_values_to_be_in_set:
              arguments:
                value_set: ['East', 'West', 'Central', 'North', 'South']
      - name: sales_qty
        tests:
          # RANGE: quantities are never negative.
          - dbt_expectations.expect_column_values_to_be_between:
              arguments:
                min_value: 0
                strictly: false
      - name: sales_amt
        tests:
          - dbt_expectations.expect_column_values_to_not_be_null
          # TYPE: the measure really is numeric.
          - dbt_expectations.expect_column_values_to_be_of_type:
              arguments:
                column_type: number

  - name: rpt_category_vs_target
    description: "Actual sales against the planning target from the category_targets seed."
    columns:
      - name: category_name
        tests:
          - unique
          - not_null
      - name: target_status
        tests:
          - accepted_values:
              arguments:
                values: ['MET', 'MISSED']
      - name: pct_of_target
        tests:
          # A sanity band: a category at 0% or 10,000% of target means the join
          # or the seed is wrong, not that the business had a great quarter.
          - dbt_expectations.expect_column_values_to_be_between:
              arguments:
                min_value: 0
                max_value: 1000

  - name: rpt_category_pivot
    description: "Sales amount per store, one column per category (generated by a Jinja macro)."
    columns:
      - name: store_key
        tests:
          - unique
          - not_null
""")
print("wrote", p)

Four kinds of expectation: **shape** (row count in a band), **set** (region is one of five), **range** (quantities never negative; attainment within a sane band), and **type**. These catch the wrong-but-not-null defects — a report that silently returns zero rows, or a target join producing 10,000% attainment.

### Question 36

Write the MetricFlow **time spine** — a dense, gap-free calendar.

In [ ]:
p = Path(PROJ) / "models/semantic/metricflow_time_spine.sql"
p.parent.mkdir(parents=True, exist_ok=True)
p.write_text("""-- TIME SPINE — a dense, gap-free list of dates, one row per day.
--
-- MetricFlow REQUIRES one. Without it, `dbt parse` fails with "The semantic
-- layer requires a time spine model with granularity DAY or smaller".
--
-- Why a spine and not dim_date: dim_date only contains days that actually
-- appear in sales, so it has holes (days with no transactions). Cumulative and
-- time-windowed metrics — sales_rolling_28d, sales_to_date — must be able to
-- land on EVERY day, including the ones with no sales, or a rolling window
-- silently skips them and the trend is wrong.
--
-- dbt_utils.date_spine generates the rows. The range covers the sales data
-- (2009-01-01 to 2012-12-30) with room either side.
--
-- It is registered as the spine in sem_models.yml via the time_spine property.

{{ config(materialized='table') }}

with days as (

    {{ dbt_utils.date_spine(
        datepart="day",
        start_date="cast('2008-01-01' as date)",
        end_date="cast('2015-01-01' as date)"
       )
    }}

)

select
    cast(date_day as date) as date_day
from days
""")
print("wrote", p)

Without it the project fails to parse: *"the semantic layer requires a time spine model with granularity DAY or smaller"*. `dim_date` will not do — it only holds days that appear in sales, so a rolling window landing on a missing day would silently skip it.

### Question 37

Write the semantic models — entities, dimensions and measures over the star
schema.

In [ ]:
p = Path(PROJ) / "models/semantic/sem_models.yml"
p.parent.mkdir(parents=True, exist_ok=True)
p.write_text("""# SEMANTIC MODELS — the MetricFlow layer over the star schema.
#
# A semantic model describes ONE dbt model in three parts:
#
#   entities    the keys. `primary` on a dimension, `foreign` on the fact —
#               MetricFlow uses matching entity NAMES to work out the joins, so
#               you never write a JOIN again.
#   dimensions  what you group/filter by: categorical, or time.
#   measures    the aggregatable columns, each with its aggregation.
#
# Metrics (sem_metrics.yml) are then defined on top of the measures. The payoff:
# "total sales by region by month" is a metric + two dimensions, resolved
# against the star schema automatically, instead of a hand-written join that
# every analyst re-implements slightly differently.

# The TIME SPINE registration. MetricFlow needs a dense day-grain calendar to
# anchor cumulative and windowed metrics; this points it at the spine model.
models:
  - name: metricflow_time_spine
    description: "Dense one-row-per-day calendar backing all time-based metrics."
    time_spine:
      standard_granularity_column: date_day
    columns:
      - name: date_day
        granularity: day

semantic_models:

  # ------------------------------------------------------------------ the fact
  - name: sales
    description: "Sales facts at (date, product version, store) grain."
    model: ref('fct_sales')
    # A semantic model that has dimensions must declare a PRIMARY entity. The
    # fact has no single natural key column (its grain is the three-part
    # combination), so `primary_entity` names a conceptual one instead of
    # pointing at a column. Without it: "contains dimensions, but it does not
    # define a primary entity".
    primary_entity: sale
    defaults:
      agg_time_dimension: sale_date
    entities:
      # Foreign keys into the dimensions. The NAMES must match the primary
      # entities below — that is the join.
      - name: product
        type: foreign
        expr: product_sk
      - name: store
        type: foreign
        expr: store_sk
    dimensions:
      - name: sale_date
        type: time
        expr: full_date
        type_params:
          time_granularity: day
    measures:
      - name: sales_amount
        description: "Gross sales amount."
        agg: sum
        expr: sales_amt
      - name: sales_quantity
        agg: sum
        expr: sales_qty
      - name: sales_cost_amount
        agg: sum
        expr: sales_cost
      - name: sales_margin_amount
        agg: sum
        expr: sales_mgrn
      - name: order_line_count
        description: "Number of fact rows — the count of aggregated order lines."
        agg: count
        expr: 1

  # ------------------------------------------------------- the Type 6 dimension
  - name: product
    description: "Product dimension (Type 6). Only the current version is exposed."
    model: ref('dim_product_t6')
    entities:
      - name: product
        type: primary
        expr: product_sk
    dimensions:
      # The Type 1 columns are what you normally slice by: they re-state all
      # history under today's value, so a category rename does not split a trend.
      - name: category
        type: categorical
        expr: current_category_name
      - name: subcategory
        type: categorical
        expr: current_subcategory_name
      # The Type 2 column, for "as was" reporting.
      - name: category_at_the_time
        type: categorical
        expr: category_name
      - name: brand
        type: categorical
        expr: brand_name
      - name: is_current_version
        type: categorical
        expr: is_current

  # ------------------------------------------------------- the seeded dimension
  - name: store
    description: "Store dimension, sourced from the store_master seed."
    model: ref('dim_store')
    entities:
      - name: store
        type: primary
        expr: store_sk
    dimensions:
      - name: region
        type: categorical
      - name: country
        type: categorical
      - name: store_name
        type: categorical
""")
print("wrote", p)

Entities *are* the joins: `product` and `store` are foreign on the fact and primary on the dimensions, so MetricFlow resolves the star without you writing a JOIN. Note the product model exposes both `category` (Type 1) and `category_at_the_time` (Type 2) — the semantic layer is where Type 6's two views become a user-facing choice.

### Question 38

Write the metrics — simple, ratio, derived, cumulative and filtered.

In [ ]:
p = Path(PROJ) / "models/semantic/sem_metrics.yml"
p.parent.mkdir(parents=True, exist_ok=True)
p.write_text("""# METRICS — the business definitions, written ONCE.
#
# This is the point of the semantic layer. "Margin rate" is defined here, in
# version control, tested and reviewed. Every consumer — BI tool, notebook, saved
# query — gets the same number, instead of five dashboards each dividing slightly
# different columns.
#
# The metric types:
#   simple      one measure
#   ratio       numerator / denominator, joined correctly for you
#   derived     an expression over other metrics
#   cumulative  a running or windowed total over the time dimension

metrics:

  # ------------------------------------------------------------------- simple
  - name: total_sales
    label: "Total Sales"
    description: "Gross sales amount."
    type: simple
    type_params:
      measure: sales_amount

  - name: total_quantity
    label: "Units Sold"
    type: simple
    type_params:
      measure: sales_quantity

  - name: total_cost
    label: "Total Cost"
    type: simple
    type_params:
      measure: sales_cost_amount

  - name: total_margin
    label: "Total Margin"
    type: simple
    type_params:
      measure: sales_margin_amount

  - name: order_lines
    label: "Order Lines"
    type: simple
    type_params:
      measure: order_line_count

  # -------------------------------------------------------------------- ratio
  - name: margin_rate
    label: "Margin Rate"
    description: "Margin as a share of sales. Defined once, so every report agrees."
    type: ratio
    type_params:
      numerator: total_margin
      denominator: total_sales

  - name: avg_line_value
    label: "Average Line Value"
    description: "Sales amount per aggregated order line."
    type: ratio
    type_params:
      numerator: total_sales
      denominator: order_lines

  # ------------------------------------------------------------------ derived
  - name: gross_profit
    label: "Gross Profit"
    description: "Sales minus cost — computed from other metrics, not from raw columns."
    type: derived
    type_params:
      expr: total_sales - total_cost
      metrics:
        - name: total_sales
        - name: total_cost

  # --------------------------------------------------------------- cumulative
  - name: sales_rolling_28d
    label: "Sales (rolling 28 days)"
    description: "Trailing 28-day sales — a window, handled by the semantic layer."
    type: cumulative
    type_params:
      measure: sales_amount
      cumulative_type_params:
        window: 28 days

  - name: sales_to_date
    label: "Sales To Date"
    description: "Running total of sales from the beginning of time."
    type: cumulative
    type_params:
      measure: sales_amount

  # ---------------------------------------------------------- filtered metric
  - name: sales_east_region
    label: "Sales — East Region"
    description: "A metric carrying its own filter, so the definition cannot be misapplied."
    type: simple
    type_params:
      measure: sales_amount
    filter: "{{ Dimension('store__region') }} = 'East'"
""")
print("wrote", p)

`margin_rate` stops being a formula five dashboards each re-implement and becomes one reviewed definition. `gross_profit` is derived from other metrics; `sales_rolling_28d` is a window over the time spine.

### Question 39

Write the saved queries whose exports become **extra data marts**.

In [ ]:
p = Path(PROJ) / "models/semantic/saved_queries.yml"
p.parent.mkdir(parents=True, exist_ok=True)
p.write_text("""# SAVED QUERIES — reusable metric queries that can be EXPORTED as tables.
#
# This is how the semantic layer produces EXTRA DATA MARTS without writing more
# SQL. A saved query names metrics + group-bys once; its `exports` tell dbt to
# materialize the result into the MARTS schema when you run
# `dbt build --select saved_query:*` (or `dbt sl export`).
#
# Compare with the hand-written models in models/marts/: those spell out the
# joins and aggregation. These are declared from metric definitions, so they
# cannot drift from the agreed numbers — change margin_rate once and every mart
# below follows.

saved_queries:

  - name: sales_by_region_monthly
    description: "Regional performance by month — the executive view."
    query_params:
      metrics:
        - total_sales
        - total_margin
        - margin_rate
        - order_lines
      group_by:
        - Dimension('store__region')
        - TimeDimension('sales__sale_date', 'month')
    exports:
      - name: mart_sales_by_region_monthly
        config:
          export_as: table
          schema: marts

  - name: sales_by_category_monthly
    description: "Category performance by month, using the Type 1 (current) category."
    query_params:
      metrics:
        - total_sales
        - total_quantity
        - gross_profit
        - avg_line_value
      group_by:
        - Dimension('product__category')
        - TimeDimension('sales__sale_date', 'month')
    exports:
      - name: mart_sales_by_category_monthly
        config:
          export_as: table
          schema: marts

  - name: brand_region_daily
    description: "Brand x region daily sales, with the trailing 28-day window."
    query_params:
      metrics:
        - total_sales
        - sales_rolling_28d
      group_by:
        - Dimension('product__brand')
        - Dimension('store__region')
        - TimeDimension('sales__sale_date', 'day')
    exports:
      - name: mart_brand_region_daily
        config:
          export_as: table
          schema: marts
""")
print("wrote", p)

Each names metrics plus group-bys; the `exports` block materializes the result into `MARTS`. Three more marts declared from metric definitions — so they cannot drift from the agreed numbers, unlike a hand-written copy.

### Question 40

Write the exposure and the analysis, then list every file you have created.

In [ ]:
p = Path(PROJ) / "models/exposures.yml"
p.parent.mkdir(parents=True, exist_ok=True)
p.write_text("""# EXPOSURES document a downstream USE of the data — a dashboard, a report, an ML
# model. They build nothing; they declare "this dashboard depends on these
# models", which puts it on the lineage graph and in the docs site. Then
# `dbt ls --select +exposure:sales_dashboard` shows everything a change would
# affect, before you make it.

version: 2

exposures:
  - name: sales_dashboard
    label: "Regional Sales Dashboard"
    type: dashboard
    maturity: high
    url: https://bi.example.com/dashboards/regional-sales
    description: "Regional sales, target attainment and the category pivot."
    depends_on:
      - ref('rpt_sales_by_region')
      - ref('rpt_category_vs_target')
      - ref('rpt_category_pivot')
    owner:
      name: Analytics Team
      email: analytics@example.com
""")
print("wrote", p)

An exposure documents a dashboard so it appears in the lineage graph — then `ls --select +exposure:sales_dashboard` answers "what would breaking this affect?"

### Question 41

Write the analysis, then inventory the whole project.

In [ ]:
p = Path(PROJ) / "analyses/top_products.sql"
p.parent.mkdir(parents=True, exist_ok=True)
p.write_text("""-- An ANALYSIS: a .sql file dbt COMPILES (so ref() and Jinja resolve) but never
-- runs as part of `dbt run`/`dbt build`, and never materializes. Use it for
-- ad-hoc questions you still want version-controlled and templated.
--
-- `dbt compile --select top_products` writes the resolved SQL to
-- target/compiled/... — paste that into the Snowflake console to run it.

select
    p.prod_key,
    p.current_prod_name,
    p.current_category_name,
    sum(f.sales_amt) as total_sales
from {{ ref('fct_sales') }} f
join {{ ref('dim_product_t6') }} p on f.product_sk = p.product_sk
group by 1, 2, 3
order by total_sales desc
limit 20
""")

files = sorted(f.relative_to(PROJ) for f in Path(PROJ).rglob("*") if f.is_file())
print(f"{len(files)} files in the project\n")
for f in files:
    print("  ", f)

An analysis is compiled but never run — version-controlled, `ref()`-aware, not materialized. The inventory is what gets uploaded next.

## PART E — hand the project to Snowflake

The files exist in the notebook. Now they become a **native Snowflake object**:
upload to a stage, create a `DBT PROJECT`, and execute dbt commands as SQL.

> **The packages problem.** `dbt deps` fetches dbt_utils and dbt_expectations from
> the internet, and code inside Snowflake has no outbound network by default.
> Either vendor `dbt_packages/` into the upload, or have an ACCOUNTADMIN create a
> network rule + external access integration for `hub.getdbt.com`. Question 41
> shows both.

### Question 42

Upload every project file to the stage, preserving the folder structure.

In [ ]:
import os

n = 0
for f in sorted(Path(PROJ).rglob("*")):
    if not f.is_file():
        continue
    rel = f.relative_to(PROJ)
    # PUT needs the DESTINATION directory, so rebuild the tree under demo/
    dest = f"@{STAGE}/demo/{rel.parent}".rstrip("/.")
    session.file.put(str(f), dest, auto_compress=False, overwrite=True)
    n += 1

print(f"uploaded {n} files")
session.sql(f"LIST @{STAGE}").show(50)

`session.file.put` is the notebook's way onto a stage — no SnowSQL needed. `auto_compress=False` matters: dbt must read plain `.sql` and `.yml`, not `.gz`. Check the `LIST` output has `demo/dbt_project.yml` at the top of the tree.

### Question 43

Create the dbt project object from the stage.

In [ ]:
USE DATABASE DEMO_DB;

CREATE OR REPLACE DBT PROJECT DEMO_DB.PUBLIC.SALES_DBT
    FROM @DEMO_DB.RAW.DBT_PROJECT_STAGE/demo/;

SHOW DBT PROJECTS IN DATABASE DEMO_DB;

-- If `deps` cannot reach the internet, an ACCOUNTADMIN can grant egress:
--
--   CREATE OR REPLACE NETWORK RULE dbt_hub_rule
--       MODE = EGRESS TYPE = HOST_PORT
--       VALUE_LIST = ('hub.getdbt.com', 'codeload.github.com');
--
--   CREATE OR REPLACE EXTERNAL ACCESS INTEGRATION dbt_hub_integration
--       ALLOWED_NETWORK_RULES = (dbt_hub_rule) ENABLED = TRUE;
--
-- The alternative -- run `dbt deps` on a laptop and upload the resulting
-- dbt_packages/ folder with the project -- needs no network at all and pins the
-- versions by construction. That is the predictable choice for a classroom.

The project is now a database object you can grant on, version, and schedule — the same project that runs on a laptop, running inside Snowflake.

### Question 44

Install the packages and build everything: seeds, snapshots, models and tests.

In [ ]:
EXECUTE DBT PROJECT DEMO_DB.PUBLIC.SALES_DBT ARGS = 'deps';

EXECUTE DBT PROJECT DEMO_DB.PUBLIC.SALES_DBT ARGS = 'build';

`ARGS` takes the same flags as the CLI. `build` runs seeds → snapshots → models → tests in dependency order and **stops a downstream model when an upstream test fails**, so bad data does not propagate. That is why it is the command you schedule.

### Question 45

Build one layer at a time, and run a single model — the same selectors as the
CLI.

In [ ]:
EXECUTE DBT PROJECT DEMO_DB.PUBLIC.SALES_DBT ARGS = 'run --select staging';
EXECUTE DBT PROJECT DEMO_DB.PUBLIC.SALES_DBT ARGS = 'run --select edw';
EXECUTE DBT PROJECT DEMO_DB.PUBLIC.SALES_DBT ARGS = 'run --select marts';

EXECUTE DBT PROJECT DEMO_DB.PUBLIC.SALES_DBT ARGS = 'test --select dim_product_t6';

-- Rebuild incrementals from scratch, ignoring the is_incremental() cutoff.
-- Needed when a model's SQL or schema changes.
EXECUTE DBT PROJECT DEMO_DB.PUBLIC.SALES_DBT ARGS = 'build --full-refresh';

Because each layer is a folder, the folder name doubles as a layer selector. Building layer by layer is how you debug a broken DAG without waiting for the whole thing.

### Question 46

Materialize the semantic layer's saved queries — the extra marts.

In [ ]:
EXECUTE DBT PROJECT DEMO_DB.PUBLIC.SALES_DBT ARGS = 'build --select saved_query:*';

SHOW TABLES IN SCHEMA DEMO_DB.MARTS;

Three more marts, declared from metric definitions rather than written as SQL. Note the `mf` command-line tool (`mf query`, `mf list metrics`) is a *local* utility and is not available inside Snowflake — the semantic models still parse and the exports still build.

## PART F — lineage across the layers

dbt knows the whole graph because every model declares its inputs with `ref()`
and `source()`. Both directions are answerable — and the second one is the
question to ask *before* editing anything.

### Question 47

Trace **upstream**: everything that feeds the regional sales report.

In [ ]:
EXECUTE DBT PROJECT DEMO_DB.PUBLIC.SALES_DBT
    ARGS = 'ls --select +rpt_sales_by_region';

The `+` prefix means "and everything upstream". You should see the full chain — sources, the snapshot, staging, seeds, and the edw dimensions and fact: raw → stg → edw → marts in one list.

### Question 48

Trace **downstream**: the blast radius if `stg_sales` changed.

In [ ]:
EXECUTE DBT PROJECT DEMO_DB.PUBLIC.SALES_DBT
    ARGS = 'ls --select stg_sales+';

-- And what a dashboard depends on:
EXECUTE DBT PROJECT DEMO_DB.PUBLIC.SALES_DBT
    ARGS = 'ls --select +exposure:sales_dashboard';

A trailing `+` walks the other way. This is the payoff of never hardcoding a table name: the graph can answer impact questions that a folder of numbered SQL scripts cannot.

### Question 49

Confirm the layers landed where the routing said they would.

In [ ]:
SELECT table_schema, table_name, table_type
FROM DEMO_DB.INFORMATION_SCHEMA.TABLES
WHERE table_schema IN ('RAW','STG','EDW','MARTS','SEED')
ORDER BY CASE table_schema
             WHEN 'RAW' THEN 1 WHEN 'SEED' THEN 2 WHEN 'STG' THEN 3
             WHEN 'EDW' THEN 4 ELSE 5 END,
         table_name;

The physical proof of the lineage: staging as **views**, edw and marts as **tables**, each in its own schema — exactly the routing set in `dbt_project.yml`.

## PART G — inspect what you built

### Question 50

Look at the Type 6 dimension: all three views on one row.

In [ ]:
SELECT product_sk, prod_key,
       category_name           AS type2_as_of_then,
       current_category_name   AS type1_today,
       previous_category_name  AS type3_changed_from,
       valid_from, valid_to, is_current
FROM DEMO_DB.EDW.DIM_PRODUCT_T6
ORDER BY prod_key, valid_from
LIMIT 20;

-- The surrogate key is unique per VERSION; prod_key is not.
SELECT COUNT(*) AS rows,
       COUNT(DISTINCT product_sk) AS distinct_sks,
       COUNT(DISTINCT prod_key)   AS distinct_products
FROM DEMO_DB.EDW.DIM_PRODUCT_T6;

On a first build every product has one version, so all three columns agree and `rows = distinct_sks = distinct_products`. Question 51 changes a product and they diverge — which is the whole point of Type 6.

### Question 51

Query the star: the fact joined to all three dimensions.

In [ ]:
SELECT d.year_num,
       d.month_name,
       s.region,
       p.current_category_name AS category,
       SUM(f.sales_amt) AS sales_amt,
       SUM(f.sales_qty) AS sales_qty
FROM DEMO_DB.EDW.FCT_SALES f
JOIN DEMO_DB.EDW.DIM_DATE       d ON f.date_key   = d.date_key
JOIN DEMO_DB.EDW.DIM_PRODUCT_T6 p ON f.product_sk = p.product_sk
JOIN DEMO_DB.EDW.DIM_STORE      s ON f.store_sk   = s.store_sk
GROUP BY 1, 2, 3, 4
ORDER BY 1, 2, 5 DESC
LIMIT 20;

-- Orphan check: every fact row must find its dimensions.
SELECT COUNT(*) AS orphan_rows
FROM DEMO_DB.EDW.FCT_SALES f
LEFT JOIN DEMO_DB.EDW.DIM_PRODUCT_T6 p ON f.product_sk = p.product_sk
WHERE p.product_sk IS NULL;

Four tables, three joins, all on surrogate keys — that is a star schema doing its job. `orphan_rows` must be 0; the `relationships` tests assert the same thing automatically on every build.

### Question 52

Compare the hand-written mart with the semantic-layer export.

In [ ]:
SELECT * FROM DEMO_DB.MARTS.RPT_CATEGORY_VS_TARGET ORDER BY pct_of_target DESC;

SELECT * FROM DEMO_DB.MARTS.RPT_SALES_BY_REGION LIMIT 10;

-- Same question, two routes: SQL you wrote vs metrics you declared.
-- If these disagree, the hand-written model has drifted from the agreed
-- definition -- which is the argument for the semantic layer.
SELECT 'hand-written'      AS source, ROUND(SUM(sales_amt))  AS sales
FROM DEMO_DB.MARTS.RPT_SALES_BY_REGION
UNION ALL
SELECT 'metricflow export', ROUND(SUM(total_sales))
FROM DEMO_DB.MARTS.MART_SALES_BY_REGION_MONTHLY;

The comparison is the lesson: a hand-written mart and a declared one should agree, and when they stop agreeing it is almost always the SQL that drifted.

### Question 53

Make history actually happen: change a product, re-snapshot, rebuild, and watch
the Type 6 columns diverge.

In [ ]:
-- 1. Change a product in the source.
UPDATE DEMO_DB.RAW.PRODUCT
SET CATEGORY_NAME = 'category-5'
WHERE PROD_KEY = (SELECT MIN(PROD_KEY) FROM DEMO_DB.RAW.PRODUCT);

-- 2. Capture it and rebuild (run these, then re-run the query below).
EXECUTE DBT PROJECT DEMO_DB.PUBLIC.SALES_DBT ARGS = 'snapshot';
EXECUTE DBT PROJECT DEMO_DB.PUBLIC.SALES_DBT ARGS = 'build';

-- 3. Two rows now, for the same product.
SELECT product_sk, prod_key,
       category_name          AS type2_as_of_then,
       current_category_name  AS type1_today,
       previous_category_name AS type3_changed_from,
       valid_from, valid_to, is_current
FROM DEMO_DB.EDW.DIM_PRODUCT_T6
WHERE prod_key = (SELECT MIN(PROD_KEY) FROM DEMO_DB.RAW.PRODUCT)
ORDER BY valid_from;

This is the payoff. The old row keeps its original `category_name` (Type 2) but gets a `valid_to` and `is_current = false`; **both** rows now show `category-5` in `current_category_name` (Type 1); and the new row's `previous_category_name` names what it changed from (Type 3). One dimension, three questions answered.

## PART H — scheduling

A project is only useful if it runs without you. Inside Snowflake the scheduler
is already there — a `TASK`, no cron and no external orchestrator.

### Question 54

Create a task that builds the project every morning, and start it.

In [ ]:
CREATE OR REPLACE TASK DEMO_DB.PUBLIC.SALES_DBT_DAILY
    WAREHOUSE = COMPUTE_WH
    SCHEDULE = 'USING CRON 0 6 * * * UTC'
AS
    EXECUTE DBT PROJECT DEMO_DB.PUBLIC.SALES_DBT ARGS = 'build';

-- Tasks are created SUSPENDED.
ALTER TASK DEMO_DB.PUBLIC.SALES_DBT_DAILY RESUME;

SHOW TASKS IN SCHEMA DEMO_DB.PUBLIC;

Change `COMPUTE_WH` if your warehouse is named differently. Because the task runs `build`, a failing test stops the run and the failure surfaces in task history rather than silently publishing bad data.

### Question 55

Check the task history — how you find out a scheduled run failed.

In [ ]:
SELECT name, state, scheduled_time, completed_time, error_message
FROM TABLE(DEMO_DB.INFORMATION_SCHEMA.TASK_HISTORY(
        TASK_NAME => 'SALES_DBT_DAILY'))
ORDER BY scheduled_time DESC
LIMIT 20;

-- Suspend it when you are done experimenting, so it stops consuming credits.
-- ALTER TASK DEMO_DB.PUBLIC.SALES_DBT_DAILY SUSPEND;

`state` and `error_message` are what monitoring watches. Remember to **suspend the task** when you finish the lab — a resumed task keeps running on schedule and keeps spending credits.

---

You have built the full path from raw to data marts: sources, staging, snapshots, a Type 6 dimension, a star schema on surrogate keys, marts, quality tests, unit tests, a semantic layer, lineage, and a schedule — all inside Snowflake, from one notebook, with no credentials anywhere.

## PART I — what dbt actually created, and cleaning up

dbt wrote SQL on your behalf. Snowflake will show you exactly what, which is the
best way to close the loop: you wrote a `SELECT`, and *this* is the object that
came out.

### Question 56

Ask Snowflake for the DDL of the objects dbt built. Compare a view, a table
and the fact.

In [ ]:
USE DATABASE DEMO_DB;

-- A staging model: dbt wrapped your SELECT in CREATE VIEW.
SELECT GET_DDL('VIEW', 'DEMO_DB.STG.STG_SALES') AS staging_view_ddl;

-- A dimension: same SELECT idea, but materialized as a table.
SELECT GET_DDL('TABLE', 'DEMO_DB.EDW.DIM_PRODUCT_T6') AS type6_dimension_ddl;

-- The fact, built incrementally.
SELECT GET_DDL('TABLE', 'DEMO_DB.EDW.FCT_SALES') AS fact_ddl;

-- Or a whole layer at once.
SELECT GET_DDL('SCHEMA', 'DEMO_DB.EDW') AS whole_edw_layer_ddl;

This is the punchline of the whole day: you never wrote `CREATE TABLE`, yet here is the exact DDL. The materialization decided the shape — `STG_SALES` came out a **view**, `DIM_PRODUCT_T6` a **table** — from one config line, not from you writing two different statements.

### Question 57

Show every object the lab created, grouped by layer, with row counts.

In [ ]:
-- The full inventory, in dependency order.
SELECT table_schema, table_name, table_type, row_count, bytes
FROM DEMO_DB.INFORMATION_SCHEMA.TABLES
WHERE table_schema IN ('RAW','SEED','STG','EDW','MARTS')
ORDER BY CASE table_schema
             WHEN 'RAW' THEN 1 WHEN 'SEED' THEN 2 WHEN 'STG' THEN 3
             WHEN 'EDW' THEN 4 ELSE 5 END,
         table_name;

-- The non-table objects too.
SHOW STAGES IN DATABASE DEMO_DB;
SHOW FILE FORMATS IN DATABASE DEMO_DB;
SHOW DBT PROJECTS IN DATABASE DEMO_DB;
SHOW TASKS IN DATABASE DEMO_DB;

Read it top to bottom and you can see the pipeline: two raw tables in, two seeds, staging views, four EDW tables forming the star, and the marts. `row_count` is null for views — they store nothing, which is exactly why staging is materialized that way.

### Question 58

**Tear it down.** Everything the lab created, in one cell — so you can re-run
the class from scratch, and so nothing keeps billing.

In [ ]:
-- 1. STOP THE CLOCK FIRST. A resumed task keeps running on schedule.
ALTER TASK IF EXISTS DEMO_DB.PUBLIC.SALES_DBT_DAILY SUSPEND;

-- 2. Drop the lab objects.
DROP TASK        IF EXISTS DEMO_DB.PUBLIC.SALES_DBT_DAILY;
DROP DBT PROJECT IF EXISTS DEMO_DB.PUBLIC.SALES_DBT;

-- 3. The database takes the schemas, tables, views, stages and file format
--    with it. Comment this out if you want to keep the results.
DROP DATABASE IF EXISTS DEMO_DB;

-- 4. The warehouse. Suspending is usually enough; drop only if you made it
--    for this lab.
ALTER WAREHOUSE IF EXISTS COMPUTE_WH SUSPEND;
-- DROP WAREHOUSE IF EXISTS COMPUTE_WH;

-- 5. And the role, if you created one.
-- USE ROLE ACCOUNTADMIN;
-- DROP ROLE IF EXISTS DBT_LAB_ROLE;

SHOW DATABASES LIKE 'DEMO_DB';

Order matters: **suspend the task before dropping anything it reads**, or a scheduled run can fire against half-deleted objects. Dropping the database is the one statement that removes the most — schemas, tables, views, stages, the file format and the seeds all go with it. Snowflake keeps it in Time Travel for the retention period, so an accidental drop is recoverable with `UNDROP DATABASE DEMO_DB`.